# TRACE cue tagger prototype

Module 1 of *Bridging TRACE and DP-Fusion for Implicit PII*: `X_priv = NER(D) ∪ V_cot ∪ V_att`.
Three signal sources each return char-offset spans marking text that leaks a target attribute,
and this notebook runs them one at a time with **every intermediate visible** — the prompt that
went out, the raw model text that came back, the quotes pulled from it, the attention mass per
word — rather than just the spans they collapse to.

That visibility is the point. The sources fail in different, quiet ways: the attacker can
refuse or drift off-format, the chain can quote text it never actually read, attention can
concentrate on punctuation. All of those reduce to "fewer spans" if you only look at the output.

| source | what it is | needs |
| --- | --- | --- |
| `NER(D)` | spaCy `en_core_web_trf`, deliberately not LLM-prompted so the NER-vs-cues ablation is not circular | spaCy model |
| `V_cot` | adversarial inference → privacy-leakage chain → quoted evidence | LLM, 2 calls/attribute |
| `V_att` | top-K words by last-layer attention from the final token | LLM, 1 forward pass |

**Kernel**: `Python (dpfusion-repro)`.
**Hardware**: Llama-2-7b-chat in fp16 needs ~13.5 GB and a 24 GB card is enough for one copy.
**Note**: the implementation lives in `fusit.trace`, which is gitignored — this
notebook is committed but its dependency is not. See the closing section.

## 1. Setup

In [ ]:
import textwrap, time

import torch

from fusit.dataset import get_dataset
from fusit.utils import find_phrase_offsets
from fusit import trace as ct

# The two LLM-backed sources are configured separately so they can be varied independently.
# They default to the same model, which is then loaded once and shared.
ATT_MODEL_ID = "NousResearch/Llama-2-7b-chat-hf"   # V_att: attention extraction
COT_MODEL_ID = "NousResearch/Llama-2-7b-chat-hf"   # V_cot: inference + leakage chain

DATASET = "synthetic"    # "synthpai" profiles are ~4x longer and much slower
ITEM_N  = 0              # index into the seeded sample below

print(f"torch {torch.__version__} | cuda {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}, {p.total_memory/1e9:.1f} GB")
print(f"V_att model: {ATT_MODEL_ID}")
print(f"V_cot model: {COT_MODEL_ID}")

In [ ]:
ds = get_dataset(DATASET)
item = ds.select(n=8, seed=0)[ITEM_N]
ATTRIBUTE = next(iter(item.relevant_pii))          # synthetic items have exactly one
GROUND_TRUTH = item.relevant_pii[ATTRIBUTE]
TEXT = item.text

print(f"item        : {item.username}")
print(f"attribute   : {ATTRIBUTE}  ->  prompt label {ct.ATTRIBUTE_LABEL[ATTRIBUTE]!r}")
print(f"ground truth: {GROUND_TRUTH!r}")
print(f"hardness    : {getattr(item, 'hardness', '-')}   (1 = cue stated outright, 5 = oblique)")
print(f"text        : {len(TEXT)} chars\n")
print(textwrap.fill(TEXT, 100))

Loading is the slow step (~2.5 min for a 7B from cold cache). `load` caches by model id, so
identical `ATT_MODEL_ID` / `COT_MODEL_ID` costs one load and one copy of the weights.

**Two *different* 7B models will not co-reside in 24 GB** (~13.5 GB each in fp16). If you want
to mix, call `free()` between the two halves of the notebook, or run it twice.

`attn_implementation` is left at the default on purpose: `attention_spans` flips the model to
`eager` for its single forward pass and flips it back, because loading the *whole* model eager
made DP-Fusion's batched incremental decoding emit NaN fused probabilities.

Llama-2 needs two things Qwen did not, both handled in `fusit.trace`:

- NousResearch's mirrors ship **no `tokenizer.chat_template`**, so `apply_chat_template` raises.
  `_format_chat` falls back to Meta's official `[INST] <<SYS>>...` string — the same one
  `repro/attackers.py` uses, so the prompt matches what the paper's Llama-2 columns saw.
- its context is **4096 tokens**, not 32k. `_chat` truncates from the left when the prompt plus
  the generation budget would overflow, keeping the tail where the format instructions sit.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

_loaded = {}

def load(model_id):
    """Load (or reuse) a model+tokenizer pair. Cached by id so sharing costs one copy."""
    if model_id not in _loaded:
        t0 = time.perf_counter()
        tok = AutoTokenizer.from_pretrained(model_id)
        mdl = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda:0")
        mdl.eval()
        print(f"loaded {model_id} in {time.perf_counter()-t0:.0f}s")
        _loaded[model_id] = (mdl, tok)
    return _loaded[model_id]


def free(model_id=None):
    """Drop cached weights so a different model can be loaded on the same card.

    This only releases the cache's reference. `att_model` / `cot_model` are still bound in
    the notebook namespace and keep the weights alive on their own, so this clears those
    names too -- otherwise nothing is actually freed.
    """
    for k in ([model_id] if model_id else list(_loaded)):
        _loaded.pop(k, None)
    for name in ("att_model", "att_tok", "cot_model", "cot_tok"):
        globals().pop(name, None)
    import gc; gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


att_model, att_tok = load(ATT_MODEL_ID)
cot_model, cot_tok = load(COT_MODEL_ID)

print(f"shared weights : {att_model is cot_model}")
print(f"context window : {cot_model.config.max_position_embeddings} tokens")
print(f"chat_template  : {'yes' if cot_tok.chat_template else 'no -> _format_chat fallback'}")
print(f"attn impl      : {att_model.config._attn_implementation!r}")
print(f"GPU allocated  : {torch.cuda.memory_allocated()/1e9:.1f} GB")

### A way to look at spans

Every source returns `[[start, end], ...]` into `TEXT`. This renders them inline so the
selections can be compared by eye rather than by offset arithmetic.

In [ ]:
def highlight(text, spans, width=100, mark=("[", "]")):
    """Render text with spans bracketed. Overlapping spans are merged first."""
    if not spans:
        return "(no spans)"
    merged, out, prev = ct.merge_spans(spans), [], 0
    for s, e in merged:
        out.append(text[prev:s]); out.append(mark[0] + text[s:e] + mark[1]); prev = e
    out.append(text[prev:])
    return textwrap.fill("".join(out), width)


def coverage(text, spans):
    merged = ct.merge_spans(spans)
    chars = sum(e - s for s, e in merged)
    return f"{len(merged)} spans, {chars} chars, {100*chars/len(text):.1f}% of the document"


def words_of(text, spans):
    return [text[s:e] for s, e in ct.merge_spans(spans)]

## 2. NER(D) — the baseline this project is trying to beat

`NER_BACKEND` below selects the tagger. It defaults to `presidio`, which is what DP-Fusion's
appendix A.16 uses — Microsoft Presidio driving `dslim/bert-base-NER`, reported there at
F1 85.4% against 76.1% for spaCy `en_core_web_lg`. The next section measures the two against
each other; set `NER_BACKEND = "spacy"` to run this section the old way.

Either way the point of the section is the same, and the stronger tagger makes it more
sharply: a NER only marks things that **are** entities. On an item whose attribute leaks
through word choice or local knowledge rather than a named entity, it has nothing to grab —
and that is the whole reason `V_cot` and `V_att` exist.

In [ ]:
NER_BACKEND = "presidio"        # "spacy" for the en_core_web_trf path this repo ran first
NER_OPTIONS = {}                # presidio: {"with_dates": True} to patch the DATETIME gap below

t0 = time.perf_counter()
ner = ct.ner_spans(TEXT, backend=NER_BACKEND, **NER_OPTIONS)
elapsed = time.perf_counter() - t0

if NER_BACKEND == "presidio":
    print(f"Presidio + {ct.PRESIDIO_NER_MODEL}  ({elapsed:.2f}s)")
else:
    print(f"spaCy en_core_web_trf  ({elapsed:.2f}s) | labels kept: {sorted(ct.NER_LABELS)}")

print(coverage(TEXT, ner))
print(f"caught: {words_of(TEXT, ner)}")

if NER_BACKEND == "presidio":
    for d in ct.presidio_entities(TEXT, **NER_OPTIONS):
        print(f"  {d['entity_type']:9s} <- {d['presidio_type']:14s} {d['score']:.2f}  {d['text']!r}")

print()
print(highlight(TEXT, ner))

### Why that backend

DP-Fusion's appendix A.16 does not use spaCy. It reports F1 **85.4%** for a
Presidio tagger driving `dslim/bert-base-NER` against **76.1%** for spaCy `en_core_web_lg`, and
picks the former — so `fusit.trace.ner` carries both and `ner_backend` selects.

Presidio is not just a better NER. Next to the transformer it runs pattern recognizers for the
identifier-shaped PII a CoNLL-trained model has no class for — emails, phones, cards, SSNs,
IBANs — which is most of what TAB-ECHR files under CODE.

It also has a hole. BERT-NER's labels are CoNLL's (PER/LOC/ORG/MISC), which has **no date
class**, so the presidio backend returns no DATETIME at all — while `DATE_TIME` still shows up
in `get_supported_entities()`, because the recognizer is registered and simply never fires.
DATETIME is one of the three types the paper scopes its TAB-ECHR attack to, so
`with_dates=True` unions spaCy's DATE entities back in. Off by default: it is then no longer
purely the paper's tagger.

In [ ]:
from fusit.trace import ner as ner_backends   # `ner` is the span list from above

probe = ("Dr. Sarah Chen was born on 12 March 1984 in Busan, South Korea and now works at "
         "Siemens in Zurich. Reach her at sarah.chen@example.com or +41 44 668 18 00. "
         "Her SSN is 078-05-1120 and card 4111 1111 1111 1111.")

for tag, kw in [("spacy", {}), ("presidio", {}), ("presidio +dates", {"with_dates": True})]:
    sp = ner_backends.ner_spans(probe, backend=tag.split()[0], **kw)
    print(f"{tag:16s} {len(ct.merge_spans(sp)):2d} spans  {100*ct.coverage(probe, sp):5.1f}%")
    print(f"                 {words_of(probe, sp)}")

print("\nwhat spacy misses that presidio catches:")
missed = set(map(tuple, ct.merge_spans(ner_backends.ner_spans(probe, backend="presidio")))) \
       - set(map(tuple, ct.merge_spans(ner_backends.ner_spans(probe, backend="spacy"))))
print(" ", sorted(probe[s:e] for s, e in missed))

In [ ]:
# typed detections -- entity_type is the TAB-ECHR category, so these group straight into
# DP-Fusion's per-type privacy groups (fusit.utils.ENTITY_TYPES / DEFAULT_BETA_DICT)
for d in ner_backends.presidio_entities(probe, with_dates=True):
    print(f"  {d['entity_type']:9s} <- {d['presidio_type']:14s} {d['score']:.2f}  {d['text']!r}")

print("\nthe DATETIME gap, isolated:")
t2 = "She was born on 12 March 1984 and hired in June 2019."
for wd in (False, True):
    got = [d["text"] for d in ner_backends.presidio_entities(t2, with_dates=wd) if d["entity_type"] == "DATETIME"]
    print(f"  with_dates={wd!s:5s} -> {got}")

## 3. Adversarial inference — the attacker's guess

`V_cot` starts by asking the model to profile the author. The prompt is TRACE-RPS's verbatim,
and it pins the output format so the guess can be parsed back out.

In [ ]:
label   = ct.ATTRIBUTE_LABEL[ATTRIBUTE]
options = ct.ATTRIBUTE_OPTIONS.get(ATTRIBUTE, "")

infer_prompt = ct.ADVERSARIAL_INFERENCE_QUERY_PROMPT_TEMPLATE.format(
    target_attribute=label, target_attribute_options=options, comments=TEXT
)
print("SYSTEM:", ct.ADVERSARIAL_INFERENCE_SYSTEM_PROMPT.strip())
print("\nUSER:")
print(infer_prompt.strip())

The raw completion, before any parsing:

In [ ]:
t0 = time.perf_counter()
infer_raw = ct.chat(cot_model, cot_tok, ct.ADVERSARIAL_INFERENCE_SYSTEM_PROMPT,
                     infer_prompt, max_new_tokens=400)
print(f"({time.perf_counter()-t0:.1f}s, {len(infer_raw)} chars)\n")
print(infer_raw)

Parsing is where this quietly breaks. `_parse_inference_response` strips bullets and markdown
before matching the field names, because Llama-2 indents them as `"  - Guess: ..."` — an
unstripped `startswith` dropped the guess on 82% of its cells and scored that column at 8.7%
against the paper's 53.71%. Check the fields actually came through.

In [ ]:
parsed = ct.parse_inference_response(infer_raw)
print(f"guesses  : {parsed['guesses']}")
print(f"certainty: {parsed['certainty']}  (1-5, self-reported)")
print(f"inference: {textwrap.fill(parsed['inference'], 96, subsequent_indent='           ')}")

assert parsed["guesses"], "no guess parsed -- the model drifted off-format; inspect infer_raw above"
print(f"\nground truth : {GROUND_TRUTH!r}")
print(f"top-1 guess  : {parsed['guesses'][0]!r}")
print(f"ground truth appears in the top 3: "
      f"{any(GROUND_TRUTH.lower() in g.lower() or g.lower() in GROUND_TRUTH.lower() for g in parsed['guesses'])}")

## 4. Inference chain — turning the guess into evidence spans

The guess alone says nothing about *which words* leaked. The second call asks the model to
justify its own inference step by step and quote the comments at each step; the quotes are
what become `V_cot`.

In [ ]:
chain_prompt = ct.PRIVACY_LEAKAGE_CHAIN_PROMPT_TEMPLATE.format(
    comments=TEXT, target_attribute=label,
    inference=parsed["inference"], guess="; ".join(parsed["guesses"]),
)
t0 = time.perf_counter()
chain_raw = ct.chat(cot_model, cot_tok, ct.PRIVACY_LEAKAGE_CHAIN_SYSTEM_PROMPT,
                     chain_prompt, max_new_tokens=500)
print(f"({time.perf_counter()-t0:.1f}s, {len(chain_raw)} chars)\n")
print(chain_raw)

Quotes are read off the `Evidence:` lines only, then located back in the text with
`find_phrase_offsets`. A quote the model paraphrased instead of copying simply will not be
found — `infer_and_chain` skips it silently, so the gap is invisible downstream. Here it is
made explicit.

In [ ]:
quotes = ct.extract_evidence_quotes(chain_raw)
print(f"{len(quotes)} quotes on Evidence lines:")
for q in quotes:
    found = find_phrase_offsets(TEXT, [q])
    flag = f"located x{len(found)}" if found else "NOT IN TEXT (paraphrased -> dropped)"
    print(f"  [{flag:34s}] {q[:78]!r}")

cot = find_phrase_offsets(TEXT, quotes) if quotes else []
n_lost = sum(1 for q in quotes if not find_phrase_offsets(TEXT, [q]))
print(f"\n{n_lost}/{len(quotes)} quotes dropped for not being verbatim")
print(coverage(TEXT, cot))
print()
print(highlight(TEXT, cot))

In [ ]:
# the real entry point should agree with the stage-by-stage walk above
t0 = time.perf_counter()
vcot = ct.infer_and_chain(TEXT, ATTRIBUTE, cot_model, cot_tok)
print(f"infer_and_chain: {time.perf_counter()-t0:.1f}s (both calls)")
print(f"guesses        : {vcot['guesses']}")
print(f"evidence spans : {coverage(TEXT, vcot['evidence_spans'])}")
print("\nGreedy decoding, so the guess should reproduce exactly; the chain may still differ "
      "slightly since its prompt embeds the inference text.")

## 5. Attention — `V_att`

A different signal entirely: no generation, one forward pass. The text is followed by the
attribute's question, and the attention **from the final token back to the context** is read off
the last layer, averaged over heads, and summed per word.

Two implementation details worth seeing rather than trusting:

- weights only exist under `eager` attention; `sdpa`/`flash_attention_2` return `None` for
  `out.attentions` even with `output_attentions=True`, with no error,
- a forward hook grabs only the last layer. Asking for all of them retains a
  `[1, heads, seq, seq]` tensor per layer — 6.3 GB at seq=2000 on a 28-layer model (Llama-2 is 32x32,
  so worse), which OOM-killed 352 SynthPAI generations on 24 GB cards.

In [ ]:
question = ct.ATTRIBUTE_QUESTION.get(ATTRIBUTE, f"What is their {label}?")
print(f"question appended: {question!r}\n")

t0 = time.perf_counter()
att, weights = ct.attention_spans(TEXT, question, att_model, att_tok, k=10, return_weights=True)
print(f"{time.perf_counter()-t0:.2f}s | attn impl restored to {att_model.config._attn_implementation!r}")
print(f"{len(weights)} words scored, top 10 kept")
print(coverage(TEXT, att))

The actual ranking, with the numbers behind it:

In [ ]:
ranked = sorted(weights, key=lambda w: w[3], reverse=True)
total = sum(w[3] for w in weights) or 1.0
hi = max(w[3] for w in ranked)

print(f"{'rank':>4}  {'word':<24} {'weight':>10} {'share':>7}  kept?")
print("-" * 68)
for i, (ws, we, word, wt) in enumerate(ranked[:20], 1):
    functional = ct.is_functional_word(word)
    kept = "yes" if [ws, we] in att else ("-- functional" if functional else "below top-10")
    bar = "#" * round(24 * wt / hi)
    print(f"{i:>4}  {word[:24]:<24} {wt:10.5f} {100*wt/total:6.2f}%  {kept:<13} {bar}")

Two things to read off that table before trusting `V_att`.

**The first token is an attention sink.** Expect the opening word to take a large share of the
mass — on the sample item `oh` alone took 64.9% — regardless of whether it carries any signal.
This is a known artifact of decoder attention, not a finding about that word, and it costs a slot
in the top-K. Anything ranked below a sink is competing for the remainder, so read the *relative*
ordering from rank 2 down rather than the absolute shares. If you are judging whether `V_att` is
picking real cues, compare it against `random_spans_matched`, not against nothing.

**Functional words are dropped before the top-K is taken.** They routinely soak up attention
(`the`, `is`, punctuation) and would otherwise spend the redaction budget on tokens carrying no
attribute signal. `content_word_spans` is the pool the ranking draws from, and the same pool the
E2 random control samples from — comparing cue spans against random spans of the *same size* is
the only way to separate "this word is risky" from "more text was deleted".

In [ ]:
n_func = sum(1 for _, _, w, _ in weights if ct.is_functional_word(w))
func_mass = sum(wt for _, _, w, wt in weights if ct.is_functional_word(w))
print(f"functional words : {n_func}/{len(weights)} ({100*n_func/len(weights):.0f}%)")
print(f"attention on them: {100*func_mass/total:.1f}% of the total mass")
print(f"top-5 functional : {[w for _,_,w,_ in sorted((x for x in weights if ct.is_functional_word(x[2])), key=lambda x: -x[3])[:5]]}")
print()
print(highlight(TEXT, att))

## 6. The three sources side by side

`build_x_priv` is the module's entry point: it unions the requested sources and merges
overlapping spans.

In [ ]:
sources = {"ner": ner, "att": att, "cot": cot}
print(f"{'source':<8} {'spans':>6} {'chars':>7} {'% doc':>7}  words")
print("-" * 100)
for name, sp in sources.items():
    m = ct.merge_spans(sp)
    chars = sum(e - s for s, e in m)
    print(f"{name:<8} {len(m):>6} {chars:>7} {100*chars/len(TEXT):>6.1f}%  {str(words_of(TEXT, sp))[:60]}")

union = ct.merge_spans(ner + att + cot)
uc = sum(e - s for s, e in union)
print(f"{'union':<8} {len(union):>6} {uc:>7} {100*uc/len(TEXT):>6.1f}%")

In [ ]:
# how much do the sources actually agree? (character-level overlap)
def char_set(spans):
    return {i for s, e in ct.merge_spans(spans) for i in range(s, e)}

cs = {k: char_set(v) for k, v in sources.items()}
names = list(cs)
print(f"{'':<8}" + "".join(f"{n:>10}" for n in names))
for a in names:
    row = "".join(f"{(100*len(cs[a] & cs[b])/len(cs[a]) if cs[a] else 0):>9.0f}%" for b in names)
    print(f"{a:<8}{row}")
print("\nread as: % of ROW's characters that COLUMN also selected")

In [ ]:
# build_x_priv drives att and cot through ONE model argument, so it only reproduces the
# stage-by-stage walk above when the two ids match.
assert ATT_MODEL_ID == COT_MODEL_ID, "build_x_priv takes a single model; set the two ids equal"

t0 = time.perf_counter()
x_priv = ct.build_x_priv(TEXT, [ATTRIBUTE], cot_model, cot_tok, k=10,
                         ner_backend=NER_BACKEND, ner_options=NER_OPTIONS)
print(f"build_x_priv (all three sources): {time.perf_counter()-t0:.1f}s")
print(coverage(TEXT, x_priv))
print()
print(highlight(TEXT, x_priv))

## 7. Does redacting `X_priv` actually stop the attacker?

The end-to-end question. Blank the spans out and re-run the same guess. This is a crude stand-in
for the real pipeline — DP-Fusion paraphrases rather than blanking, so treat it as an upper
bound on what redaction buys.

In [ ]:
def blank(text, spans, ph="_"):
    out, prev = [], 0
    for s, e in ct.merge_spans(spans):
        out.append(text[prev:s]); out.append(ph * (e - s)); prev = e
    out.append(text[prev:])
    return "".join(out)

print(f"{'redaction':<16} {'% doc':>7}  top-3 guesses")
print("-" * 96)
for name, sp in [("none", []), (f"ner ({NER_BACKEND})", ner), ("X_priv", x_priv)]:
    redacted = blank(TEXT, sp)
    g = ct.guess_attribute(redacted, ATTRIBUTE, cot_model, cot_tok)["guesses"]
    pct = 100 * sum(e - s for s, e in ct.merge_spans(sp)) / len(TEXT)
    print(f"{name:<16} {pct:>6.1f}%  {g[:3]}")
print(f"\nground truth: {GROUND_TRUTH!r}")

## 8. Scratch

`att_model`, `cot_model`, `TEXT`, `ATTRIBUTE` and the helpers are all live. Swap `ITEM_N` or
`DATASET` at the top and re-run to look at another item — a `synthpai` profile exercises the
long-input path that the attention hook exists for, and the 4096-token truncation in
`fusit.trace.chat`.

### Using this outside the notebook

Everything above comes from `fusit.trace`, so there is nothing notebook-specific to port:

```python
from fusit.trace import CueTagger

tagger = CueTagger(model, tokenizer, attributes=["occupation"], sources=["ner", "cot", "att"])
tagger.extract_spans(document)   # merged char offsets
tagger.explain(document)         # {source: spans} — the ablation, before the union
```

`CueTagger` also answers `extract_private_phrases`, which is the contract
`fusit.dp_fusion` already expects of a tagger, so it goes straight into DP-Fusion:

```python
dpf = DPFusion(model=model, tokenizer=tokenizer, tagger=tagger)
dpf.add_message("user", document, is_private=True)
dpf.run_tagger()                 # private/public token counts stay equal
```

Prefer `extract_spans` where the caller holds the raw document: phrases get re-matched
*everywhere* they occur in the full prompt, so a one-word attention cue like `oh` redacts every
`oh`. `fusit.trace.tagger`'s docstring has the details.

In [ ]:
free()
assert torch.cuda.memory_allocated() < 1e9, "weights still resident -- some cell holds a reference"